In [3]:
import os
import warnings
import torch
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.sparse as sp
from process import utils
from process.utils import normalize_sparse_matrix, sparse_mx_to_torch_sparse_tensor
import scanpy as sc 
plt.rcdefaults()
warnings.filterwarnings('ignore')
os.environ['R_HOME'] = r'E:/R/R-4.4.1'

In [4]:
# sample = '151672'
path = 'F:/Code/spatial_domain/SGFST/SGFST/data/Human_Breast_Cancer'
adata = sc.read_visium(path=os.path.join(path), count_file='filtered_feature_bc_matrix.h5', load_images=True)
adata.var_names_make_unique()
labels_df = pd.read_table(os.path.join(path, "metadata.tsv"), sep='\t')
labels_df.index = adata.obs.index
# adata.obs['ground_truth'] = labels_df["layer_guess_reordered"]
# adata = adata[~adata.obs['ground_truth'].isnull()].copy()

In [5]:
import numpy as np
section_id = 'V1'
input_dir = os.path.join('data/Breast_Cancer', section_id)
cell_type_indeces = np.load('D:/st-project/MuCST-data/MuCST-data/Breast_Cancer/V1/cell_types.npy')
adata.obs['Ground Truth'] = cell_type_indeces
adata.obs['Ground Truth'] = adata.obs['Ground Truth'].astype('int')
adata.obs['ground_truth'] = adata.obs['Ground Truth'].astype('category')

In [6]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.highly_variable_genes(adata, flavor='seurat_v3', n_top_genes=3000)
adata = adata[:, adata.var.highly_variable].copy()
adata.layers['counts'] = adata.X.copy()

In [7]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.scale(adata, zero_center=False, max_value=10)

In [8]:
signal_activity = utils.get_signal(adata, prior_file='F:/Code/spatial_domain/SGFST/SGFST/data/kegg/gene_sets.gmt', path=os.path.join(path), threads=1)
adata.obsm['signal'] = signal_activity.loc[adata.obs_names].values

spatial_adj, graph_nei = utils.spatial_construct_graph(adata, radius=300) # 560
signal_adj = utils.features_construct_graph(adata.obsm['signal'])

read cache time: 0.20s
Using existing signal activity file...
The graph contains 11032 undirected edges, 3798 cells.
5.8094 neighbors per cell on average.


In [7]:
features = torch.FloatTensor(adata.X.toarray())
labels = adata.obs['ground_truth'].values

spatial_adj = normalize_sparse_matrix(spatial_adj + sp.eye(adata.shape[0]))
spatial_adj = sparse_mx_to_torch_sparse_tensor(spatial_adj)

signal_adj = normalize_sparse_matrix(signal_adj + sp.eye(adata.shape[0]))
signal_adj = sparse_mx_to_torch_sparse_tensor(signal_adj)

graph_nei_tensor = torch.LongTensor(graph_nei.numpy())

In [8]:
def sym_nonneg_zero_diag(A):
    """
    对称 + 非负 + 去对角
    """
    A = 0.5 * (A + A.t())
    A = torch.clamp(A, min=0.0)
    A = A - torch.diag_embed(torch.diag(A))
    return A


def normalize_prob_graph(A, eps=1e-10):
    """
    概率图:
    P_ij = A_ij / vol(A)
    """
    A = sym_nonneg_zero_diag(A)
    vol = A.sum()
    if vol <= eps:
        n = A.size(0)
        P = torch.ones_like(A) / (n * n)
        P = P - torch.diag_embed(torch.diag(P))
        P = P / (P.sum() + eps)
        return P
    return A / (vol + eps)


def dsi_loss(A, B, eps=1e-10):
    """
    DSI(A, B) = | H(M) - 0.5*(H(A)+H(B)) |
    M = 0.5*(A/vol(A) + B/vol(B))
    """
    PA = normalize_prob_graph(A, eps=eps)
    PB = normalize_prob_graph(B, eps=eps)
    M = 0.5 * (PA + PB)

    HM = -(M * torch.log(M + eps)).sum()
    HA = -(PA * torch.log(PA + eps)).sum()
    HB = -(PB * torch.log(PB + eps)).sum()

    return torch.abs(HM - 0.5 * (HA + HB))


import torch
import torch.nn.functional as F


def huber_recon_loss(x_pred, x_true, delta=1.0):
    """
    Huber / SmoothL1 重构损失
    x_pred: decoder 输出, shape [n, p]
    x_true: 输入特征, shape [n, p]
    """
    return F.smooth_l1_loss(x_pred, x_true, beta=delta, reduction='mean')


import torch
import torch.nn.functional as F

import torch
import torch.nn.functional as F


def neighborhood_infonce_loss_fast(embd, graph_nei, temperature=0.5, eps=1e-8):
    """
    更快的全矩阵版 InfoNCE
    embd: [n, d]
    graph_nei: [n, n]，0/1邻接矩阵
    """
    device = embd.device
    n = embd.size(0)

    # 归一化
    z = F.normalize(embd, p=2, dim=1)

    # 相似度矩阵
    sim = torch.matmul(z, z.t()) / temperature  # [n, n]

    # 去掉对角线
    diag_mask = torch.eye(n, dtype=torch.bool, device=device)

    pos_mask = graph_nei.bool() & (~diag_mask)  # 正样本
    valid_mask = ~diag_mask  # 所有可参与分母的样本

    # 数值稳定：每行减去最大值
    sim = sim - sim.max(dim=1, keepdim=True)[0].detach()

    exp_sim = torch.exp(sim) * valid_mask.float()

    # 分子：正样本和
    pos_sum = (exp_sim * pos_mask.float()).sum(dim=1)  # [n]

    # 分母：除自己外所有样本
    all_sum = exp_sim.sum(dim=1) + eps  # [n]

    # 只保留有正邻居的点
    valid_nodes = pos_mask.sum(dim=1) > 0
    if valid_nodes.sum() == 0:
        return torch.tensor(0.0, device=device, requires_grad=True)

    loss = -torch.log((pos_sum[valid_nodes] + eps) / all_sum[valid_nodes])
    return loss.mean()

In [9]:
import torch
import torch.nn.functional as F


def bpr_structure_loss(scores, adj, num_neg=3, num_pos=4096):
    """
    scores: [n, n]，模型输出的 rec_adj（不要 sigmoid）
    adj:    [n, n]，0/1邻接矩阵 graph_nei
    num_neg: 每个正边采几个负样本
    num_pos: 每轮随机采多少条正边；越大越准，越小越快
    """
    device = scores.device
    n = adj.size(0)

    # bool邻接，去掉对角线
    adj_bool = (adj > 0)
    diag_mask = torch.eye(n, dtype=torch.bool, device=device)
    adj_bool = adj_bool & (~diag_mask)

    # 无向图：只取上三角正边，避免重复
    pos_idx = torch.triu(adj_bool, diagonal=1).nonzero(as_tuple=False)
    

    if pos_idx.size(0) == 0:
        return scores.sum() * 0.0

    # 随机采样部分正边，加速
    if (num_pos is not None) and (pos_idx.size(0) > num_pos):
        perm = torch.randperm(pos_idx.size(0), device=device)[:num_pos]
        pos_idx = pos_idx[perm]

    src = pos_idx[:, 0]                     # [m]
    pos = pos_idx[:, 1]                     # [m]
    m = src.size(0)

    # 随机采负样本
    neg = torch.randint(0, n, (m, num_neg), device=device)   # [m, num_neg]
    src_ex = src.unsqueeze(1)                                # [m, 1]

    # 不能采到自己，也不能采到真实邻居
    invalid = (neg == src_ex) | adj_bool[src_ex, neg]

    # 少量重采样，基本够用
    for _ in range(5):
        if not invalid.any():
            break
        neg[invalid] = torch.randint(0, n, (invalid.sum().item(),), device=device)
        invalid = (neg == src_ex) | adj_bool[src_ex, neg]

    # BPR: 希望 pos_score > neg_score
    pos_score = scores[src, pos].unsqueeze(1)    # [m, 1]
    neg_score = scores[src_ex, neg]              # [m, num_neg]

    loss = F.softplus(-(pos_score - neg_score)).mean()
    return loss

In [10]:
import random
import torch.nn as nn

def model_train(model, optimizer, features, spatial_adj, signal_adj, graph_nei, spatial_adj_dsi, signal_adj_dsi, alpha=1, beta=0.1,gama=0.05):
# def model_train(model, optimizer, features, spatial_adj, signal_adj, graph_nei, alpha=1, beta=0.1):
    np.random.seed(42)
    random.seed(42)
    torch.manual_seed(42)
    torch.cuda.manual_seed(42)

    model.train()
    optimizer.zero_grad()
    # embd, rec_adj, shared_graph,pi, disp, mean, att = model(features, spatial_adj, signal_adj)
    embd, rec_adj, pi, disp, mean, att,shared_graph = model(features, spatial_adj, signal_adj)

    structure_loss = bpr_structure_loss(rec_adj, graph_nei, num_neg=3, num_pos=4096)
    # huber_loss = huber_recon_loss(mean, features, delta=1.0)
    # infonce_loss = neighborhood_infonce_loss_fast(
    #     embd, graph_nei, temperature=0.5    )
    # dsi loss
    shared_graph = sym_nonneg_zero_diag(shared_graph)
    spatial_graph = sym_nonneg_zero_diag(spatial_adj_dsi)
    signal_graph = sym_nonneg_zero_diag(signal_adj_dsi)

    dsi_spatial = dsi_loss(shared_graph, spatial_graph)
    dsi_signal = dsi_loss(shared_graph, signal_graph)
    consistency_loss = dsi_spatial + dsi_signal
    
    zinb_loss = SGFST.ZINB(pi, theta=disp, ridge_lambda=0).loss(features, mean, mean=True)
# 建议在 optimizer.step() 后添加
    total_loss = alpha * zinb_loss + beta * structure_loss+gama*consistency_loss
   
    total_loss.backward()
    optimizer.step()

    return embd, att, total_loss.item(), pi, disp, mean, att, shared_graph
    # return embd, att, total_loss.item(), pi, disp, mean, att

In [11]:
spatial_adj_raw, graph_nei = utils.spatial_construct_graph(adata, radius=300)
signal_adj_raw = utils.features_construct_graph(adata.obsm['signal'])
# 给 DSI 用的 dense 图
spatial_adj_dsi = torch.FloatTensor(spatial_adj_raw.toarray())

if sp.issparse(signal_adj_raw):
    signal_adj_dsi = torch.FloatTensor(signal_adj_raw.toarray())
else:
    signal_adj_dsi = torch.FloatTensor(np.asarray(signal_adj_raw))

The graph contains 11032 undirected edges, 3798 cells.
5.8094 neighbors per cell on average.


In [12]:
import copy
import random
import numpy as np
import torch
from tqdm import tqdm
from model import SGFST
from sklearn.metrics import adjusted_rand_score
from sklearn.cluster import KMeans

# =========================
# 固定随机种子
# =========================
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =========================
# 设备
# =========================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

features = features.to(device)
spatial_adj = spatial_adj.to(device)
signal_adj = signal_adj.to(device)
graph_nei = graph_nei.to(device)
spatial_adj_dsi = spatial_adj_dsi.to(device)
signal_adj_dsi = signal_adj_dsi.to(device)

labels_np = np.array(labels)
n_clusters = len(np.unique(labels_np))

# =========================
# 固定最优参数
# =========================
alpha = 1
beta = 0.5
gama = 0.05
num_epochs = 500

# =========================
# 初始化模型和优化器
# =========================
model = SGFST.SGFST(
    nfeat=features.shape[1],
    nhid1=128,
    nhid2=64,
    dropout=0.1
).to(device)

optimizer = torch.optim.NAdam(
    model.parameters(),
    lr=0.001,
    weight_decay=5e-4
)

# =========================
# 保存最佳结果
# =========================
ari_max = -1
best_epoch = -1
best_emb = None
best_clusters = None
best_mean = None
best_shared_graph = None
best_model_state = None
best_loss = None

for epoch in tqdm(range(1, num_epochs + 1), leave=True, desc="Training epochs"):
    model.train()
    optimizer.zero_grad()

    embd, recon_adj, loss, pi, disp, mean, att, shared_graph = model_train(
        model,
        optimizer,
        features,
        spatial_adj,
        signal_adj,
        graph_nei,
        spatial_adj_dsi,
        signal_adj_dsi,
        alpha=alpha,
        beta=beta,
        gama=gama
    )

    loss_value = loss.item() if torch.is_tensor(loss) else float(loss)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=seed,
        n_init=20
    )
    pred_labels = kmeans.fit_predict(embd.detach().cpu().numpy())

    ari_res = adjusted_rand_score(labels_np, pred_labels)

    if (epoch % 10 == 0) or (epoch == 1) or (epoch == num_epochs):
        print(f"Epoch: {epoch:03d}, Loss: {loss_value:.4f}, ARI: {ari_res:.4f}")

    if ari_res > ari_max:
        ari_max = ari_res
        best_epoch = epoch
        best_emb = embd.detach().cpu().clone()
        best_clusters = pred_labels.copy()
        best_mean = mean.detach().cpu().clone() if torch.is_tensor(mean) else mean
        best_shared_graph = shared_graph.detach().cpu().clone() if torch.is_tensor(shared_graph) else shared_graph
        best_model_state = copy.deepcopy(model.state_dict())
        best_loss = loss_value

print(f"\n--- Training complete. Best Epoch: {best_epoch}, Best ARI: {ari_max:.4f}, Best Loss: {best_loss:.4f} ---")

Training epochs:   0%|          | 0/500 [00:00<?, ?it/s]

Epoch: 001, Loss: 1.3571, ARI: 0.2025


Training epochs:   2%|▏         | 10/500 [00:43<39:22,  4.82s/it]

Epoch: 010, Loss: 1.1875, ARI: 0.3423


Training epochs:   4%|▍         | 20/500 [01:18<28:11,  3.52s/it]

...

Epoch: 480, Loss: 0.9129, ARI: 0.5687


Training epochs:  98%|█████████▊| 490/500 [28:07<00:33,  3.31s/it]

Epoch: 490, Loss: 0.9121, ARI: 0.5774


Training epochs: 100%|██████████| 500/500 [28:40<00:00,  3.44s/it]

Epoch: 500, Loss: 0.9153, ARI: 0.5904

--- Training complete. Best Epoch: 222, Best ARI: 0.6630, Best Loss: 0.9355 ---


In [17]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.stats import pearsonr
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. 基本设置
# ============================================================
genes = ['AMFR', 'APOC1', 'GFRA1', 'IGFBP7', 'KRT17', 'LINC00645', 'SLITRK6']

domain_key = 'refined_pred'   # 也可以换成 'ground_truth'
save_dir = r"F:/Paper/SIDI/Figures/breast_maker_genes/marker_recovery_score"
os.makedirs(save_dir, exist_ok=True)

# ============================================================
# 2. 构建重构表达矩阵
#    best_mean 是模型输出的重构/去噪表达
# ============================================================
if hasattr(best_mean, "detach"):
    recon_X = best_mean.detach().cpu().numpy()
else:
    recon_X = np.asarray(best_mean)

raw_X = adata.X
if sp.issparse(raw_X):
    raw_X = raw_X.toarray()
else:
    raw_X = np.asarray(raw_X)

# 保证非负，避免后面极端值影响
raw_X = np.nan_to_num(raw_X, nan=0.0, posinf=0.0, neginf=0.0)
recon_X = np.nan_to_num(recon_X, nan=0.0, posinf=0.0, neginf=0.0)

adata.layers["reconstructed"] = recon_X

# ============================================================
# 3. 空间邻接矩阵，用于 Moran's I
# ============================================================
def to_numpy_adj(adj):
    if hasattr(adj, "detach"):
        adj = adj.detach().cpu().numpy()
    elif sp.issparse(adj):
        adj = adj.toarray()
    else:
        adj = np.asarray(adj)
    return adj

W = to_numpy_adj(graph_nei)
W = (W > 0).astype(float)
np.fill_diagonal(W, 0)

# ============================================================
# 4. 工具函数
# ============================================================
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def safe_pearson(x, y):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    if np.std(x) < 1e-12 or np.std(y) < 1e-12:
        return np.nan
    return pearsonr(x, y)[0]


def morans_i(x, W, eps=1e-12):
    x = np.asarray(x).ravel()
    z = x - np.mean(x)

    s0 = W.sum()
    if s0 < eps or np.sum(z ** 2) < eps:
        return np.nan

    n = len(x)
    return (n / s0) * (z @ W @ z) / (z @ z + eps)


def domain_specificity_score(x, domains, target_domain, eps=1e-12):
    x = np.asarray(x).ravel()
    domains = np.asarray(domains).astype(str)

    in_mask = domains == str(target_domain)
    out_mask = ~in_mask

    x_in = x[in_mask]
    x_out = x[out_mask]

    if len(x_in) == 0 or len(x_out) == 0:
        return np.nan

    mean_diff = np.mean(x_in) - np.mean(x_out)
    pooled_std = np.sqrt(0.5 * (np.var(x_in) + np.var(x_out))) + eps

    return mean_diff / pooled_std


def safe_auc(x, domains, target_domain):
    x = np.asarray(x).ravel()
    domains = np.asarray(domains).astype(str)

    y = (domains == str(target_domain)).astype(int)

    if len(np.unique(y)) < 2:
        return np.nan

    try:
        return roc_auc_score(y, x)
    except Exception:
        return np.nan


# ============================================================
# 5. 逐个 marker gene 计算 MRS
# ============================================================
domains = adata.obs[domain_key].astype(str).values

results = []

for gene in genes:
    if gene not in adata.var_names:
        print(f"Warning: {gene} not found in adata.var_names, skipped.")
        continue

    gene_idx = np.where(adata.var_names == gene)[0][0]

    x_raw = raw_X[:, gene_idx]
    x_rec = recon_X[:, gene_idx]

    # --------------------------------------------------------
    # 目标空间域：
    # 默认使用 raw 表达均值最高的 domain。
    # 如果你有明确的生物学标注，也可以手动指定 target_domain。
    # --------------------------------------------------------
    tmp = pd.DataFrame({
        "expr": x_raw,
        "domain": domains
    })

    domain_means = tmp.groupby("domain")["expr"].mean()
    target_domain = domain_means.idxmax()

    # raw 指标
    spec_raw = domain_specificity_score(x_raw, domains, target_domain)
    auc_raw = safe_auc(x_raw, domains, target_domain)
    moran_raw = morans_i(x_raw, W)

    # reconstructed 指标
    spec_rec = domain_specificity_score(x_rec, domains, target_domain)
    auc_rec = safe_auc(x_rec, domains, target_domain)
    moran_rec = morans_i(x_rec, W)

    # fidelity：重构表达与原始表达的一致性，作为辅助指标
    fidelity = safe_pearson(x_raw, x_rec)

    # 归一化后的变化量
    delta_spec = sigmoid(spec_rec) - sigmoid(spec_raw)
    delta_auc = auc_rec - auc_raw
    delta_moran = (moran_rec - moran_raw) / 2.0

    # Marker Recovery Score
    mrs = np.nanmean([delta_spec, delta_auc, delta_moran])

    results.append({
        "gene": gene,
        "target_domain": target_domain,

        "specificity_raw": spec_raw,
        "specificity_reconstructed": spec_rec,
        "delta_specificity": spec_rec - spec_raw,

        "auc_raw": auc_raw,
        "auc_reconstructed": auc_rec,
        "delta_auc": delta_auc,

        "moran_raw": moran_raw,
        "moran_reconstructed": moran_rec,
        "delta_moran": moran_rec - moran_raw,

        "raw_reconstructed_pearson": fidelity,
        "MRS": mrs
    })

mrs_df = pd.DataFrame(results)

# 按 MRS 从高到低排序
mrs_df = mrs_df.sort_values("MRS", ascending=False)

# 保存结果
out_csv = os.path.join(save_dir, "marker_recovery_score.csv")
mrs_df.to_csv(out_csv, index=False)

print(mrs_df)
print(f"Saved to: {out_csv}") 

        gene target_domain  specificity_raw  specificity_reconstructed  \
3     IGFBP7           0.0         1.042095                   3.743672   
0       AMFR          17.0         1.577861                   4.808280   
1      APOC1          19.0         0.900223                   1.804412   
2      GFRA1          16.0         2.579111                   6.385874   
6    SLITRK6          14.0         2.618161                   4.875224   
4      KRT17           9.0         1.115057                   1.402406   
5  LINC00645          18.0         1.981361                   2.051653   

   delta_specificity   auc_raw  auc_reconstructed  delta_auc  moran_raw  \
3           2.701577  0.785566           0.975860   0.190294   0.461222   
0           3.230419  0.883609           0.995995   0.112386   0.372147   
1           0.904188  0.781265           0.856705   0.075440   0.540634   
2           3.806763  0.961467           0.999003   0.037536   0.578818   
6           2.257063  0.966191  

In [ ]:
# save_dir = f'D:/st_projects/SGFST/result/DLPFC/{sample}/'
# if not os.path.exists(save_dir):
#     os.makedirs(save_dir)

adata.obs['domain'] = pd.Categorical(best_clusters)

plt.rcParams["figure.figsize"] = (5, 5)
sc.pl.spatial(
    adata,
    img_key="hires",
    color=['ground_truth', 'domain'],
    title=f'SGFST (ARI: {ari_max:.4f})',
    show=True,
    size=1.5,
)

In [15]:

import numpy as np
from scipy.spatial.distance import cdist
from collections import Counter

def refine_label(adata, radius=50, key='label'):
    old_type = np.asarray(adata.obs[key].values)
    position = np.asarray(adata.obsm['spatial'])

    # 两两距离
    distance = cdist(position, position, metric='euclidean')

    new_type = []
    for i in range(distance.shape[0]):
        # 从近到远排序，第0个是自己
        index = np.argsort(distance[i])[1:radius+1]
        neigh_type = old_type[index]

        # 多数投票
        max_type = Counter(neigh_type).most_common(1)[0][0]
        new_type.append(str(max_type))

    return new_type

In [16]:
from process.utils import BestMap

sc.set_figure_params(scanpy=True, dpi=80, dpi_save=600, frameon=True, vector_friendly=False, fontsize=12, figsize=(5, 4), color_map=None, format='pdf', facecolor=None, transparent=True, ipython_format='png2x')

new_type = refine_label(adata, 20, key='domain')
adata.obs["refined_pred"] = new_type
adata.obs["refined_pred"] = adata.obs["refined_pred"].astype('category')

adata.obs['pred'] = BestMap(pd.Categorical(adata.obs['ground_truth']).codes, pd.Categorical(adata.obs['domain']).codes)
adata.obs['pred'] = adata.obs['pred'].astype('category')

adata.obs['refined_pred'] = BestMap(pd.Categorical(adata.obs['ground_truth']).codes, pd.Categorical(adata.obs['refined_pred']).codes)
adata.obs['refined_pred'] = adata.obs['refined_pred'].astype('category')

In [17]:
ari_refine = adjusted_rand_score(adata.obs['ground_truth'], adata.obs['refined_pred'])
ari_refine

0.683561825030094